# 07 — Task Decomposition and Workflow Prompting

## Scenario
Northstar receives an email requesting a refund. Our policy states that refunds are only valid if the purchase was made within the last 30 days. We have a mock database function to check purchase dates.

**The Danger:** A naive approach tries to do everything in one prompt. This forces the LLM to hallucinate database state or policy compliance because it doesn't actually have access to the deterministic data.

In [ ]:
import os
import json
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

# Initialize the client (requires GEMINI_API_KEY environment variable)
client = genai.Client()
MODEL_ID = 'gemini-2.5-flash'

CUSTOMER_EMAIL = """
From: alice@example.com
Subject: Broken widget

Hi, my order #ORD-8812 arrived yesterday and it's completely shattered. I would like a refund please.
"""

# Our Deterministic "Database"
def check_refund_eligibility(order_id: str) -> bool:
    print(f"[SYSTEM] Checking DB for {order_id}...")
    # Mock logic: Pretend ORD-8812 is actually 60 days old and NOT eligible.
    if order_id == "ORD-8812":
        return False
    return True


## Step 1: The "Do Everything" Baseline (Anti-Pattern)

Watch what happens when we ask the LLM to handle the whole process without giving it a way to check the actual database.

In [ ]:
naive_prompt = f"""You are a customer support bot.\nRead the following email and draft a response.\nOnly approve refunds if the order is within 30 days. If you don't know the order date, guess based on the email context.\n\nEmail:\n{CUSTOMER_EMAIL}\n"""

response = client.models.generate_content(
    model=MODEL_ID,
    contents=naive_prompt,
)
print("--- Naive Response ---")
print(response.text)

# Notice: The LLM will likely approve the refund because the email says "arrived yesterday", 
# hallucinating that this means the purchase was within 30 days. It bypassed our deterministic policy!

## Step 2: The Sequential Workflow

We break the task down into a pipeline. 
1. **LLM Node:** Extract the Order ID.
2. **Deterministic Node:** Python checks the database.
3. **LLM Node:** Draft the response using the concrete DB fact.

In [ ]:
# Node 1: Extraction (Using Pydantic for a strict contract)
class ExtractedFacts(BaseModel):
    order_id: str = Field(description="The extracted order ID, or 'UNKNOWN'")
    customer_intent: str = Field(description="What the customer wants (e.g. refund, exchange)")

extract_prompt = f"""Extract the facts from the email.\nEmail:\n{CUSTOMER_EMAIL}"""

response_1 = client.models.generate_content(
    model=MODEL_ID,
    contents=extract_prompt,
    config=types.GenerateContentConfig(
        temperature=0.0,
        response_mime_type="application/json",
        response_schema=ExtractedFacts,
    )
)
facts = ExtractedFacts.model_validate_json(response_1.text)
print("--- Node 1: Extracted Facts ---")
print(facts)

# Node 2: Deterministic Policy Check
is_eligible = check_refund_eligibility(facts.order_id)
print(f"\n--- Node 2: DB Result ---")
print(f"Eligible for refund: {is_eligible}")

# Node 3: Drafting
draft_prompt = f"""You are a customer support bot.\nCustomer Intent: {facts.customer_intent}\nSystem Policy Check: Refund Eligible = {is_eligible}\n\nDraft a polite email to the customer. If they are not eligible, politely decline and state it is outside the 30-day window."""

response_3 = client.models.generate_content(
    model=MODEL_ID,
    contents=draft_prompt,
)
print("\n--- Node 3: Final Draft ---")
print(response_3.text)


## Conclusion

By decomposing the task, we prevented a hallucination, isolated the deterministic logic (the database check) from the fuzzy logic (reading/writing emails), and created observable trace points (we know exactly what Order ID was extracted).